<a href="https://colab.research.google.com/github/sweeden-ttu/profile/blob/stable/notebooks/usd-llm/01_usd_corpus_and_usdfid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · USD corpus, corruption families and the `usdfid` metric

CPU only. This notebook builds the **repair-task** data (task O1 in the plan) and the layered fidelity score the challenge would use. Everything here runs on a free Colab CPU runtime in a few minutes.

Plan and context: <https://scottweeden.online/projects/usd-scene-llm/>

In [ ]:
!pip -q install usd-core numpy tokenizers matplotlib
!wget -q -O usdlab.py https://raw.githubusercontent.com/sweeden-ttu/profile/stable/notebooks/usd-llm/usdlab.py
import usdlab as U, json, numpy as np
from pxr import Usd
print('USD', Usd.GetVersion())

## 1. A licence-clean reference stage
`build_toy_knee()` writes a procedural stand-in for the Archetypal Knee (bones, cartilage, menisci) in millimetres, Z-up, with `knee:*` attributes on every mesh. Swap in the real archetype (section 5) once the pipeline works.

In [ ]:
stage = U.build_toy_knee()          # keep a reference to the stage (its layer dies with it)
ref = stage.GetRootLayer().ExportToString()
print(len(ref), 'characters'); print(ref[:800])

## 2. Corruption families
Hold out whole *families* between train and test so the task rewards generalisation, not memorised templates.

In [ ]:
for fam in U.CORRUPTIONS:
    bad, note = U.corrupt(ref, fam, seed=1)
    print(f'{fam:20s} {note}')

## 3. `usdfid`: a layered score
L0 parse/compose gate → L1 validation (UsdValidation) → L2 structure (prim-path Jaccard, types, stage metadata) → L3 attributes (world transforms, `knee:*` values) → L4 geometry (Chamfer and F-score at τ = 1% of the bounding-box diagonal).

In [ ]:
print(U.usdfid(ref, ref))
for fam in U.CORRUPTIONS:
    bad, _ = U.corrupt(ref, fam, seed=3)
    r = U.usdfid(ref, bad)
    print(f"{fam:20s} score={r['score']:.3f}  L2_meta={r['L2_metadata']:.2f}  L3={r['L3_attributes']:.3f}  F={r['L4_fscore']:.3f}")
print('unparseable ->', U.usdfid(ref, '#usda 1.0\n( not valid')['score'])

## 4. Tokenizer audit
Count tokens before choosing a base model. Per-digit tokenizers (Gemma, Qwen3, Mistral) pay heavily for inline float arrays; a payload reference keeps geometry out of the model's output.

In [ ]:
from tokenizers import Tokenizer
forms = {'full float': U.serialize(stage), '3 dp': U.serialize(stage, 3), 'integer mm': U.serialize(stage, 0), 'payload': U.serialize(stage, payload=True)}
for name, repo in {'Llama-3.x': 'unsloth/Llama-3.2-1B-Instruct', 'Gemma 4': 'unsloth/gemma-4-E4B-it', 'Qwen3': 'Qwen/Qwen3-8B'}.items():
    tok = Tokenizer.from_pretrained(repo)
    print(f'{name:10s}', {k: len(tok.encode(v).ids) for k, v in forms.items()})

## 5. SFT pairs
`{"input", "output"}` rows are the format Megatron-Bridge's `GPTSFTDatasetConfig` and most HF SFT trainers read. Families `jitter_points` are left out of the text task because payload form hides point arrays.

In [ ]:
rows = U.make_sft_pairs(700, seed=0)
test_fams = {'units', 'swap_label'}            # held-out corruption families
train = [r for r in rows if r['family'] not in test_fams]; test = [r for r in rows if r['family'] in test_fams]
for name, part in [('training.jsonl', train), ('validation.jsonl', test)]:
    with open(name, 'w') as f:
        for r in part: f.write(json.dumps({'input': r['input'], 'output': r['output']}) + '\n')
print(len(train), 'train /', len(test), 'held-out family rows')

### Optional: the real Archetypal Knee (CC BY-NC 4.0)
Requires Kaggle credentials in Colab Secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`).

In [ ]:
# import kagglehub, os
# from google.colab import userdata
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
# path = kagglehub.dataset_download('scottweeden/the-archetypal-knee')
# real = Usd.Stage.Open(f'{path}/archetype/usd/knee_archetype.usda')
# print(sum(1 for p in real.Traverse()), 'prims')